In [65]:
# loading my original traffic dataset
import pandas as pd;
import numpy as np;
traffic = pd.read_json("traffic_observations_final.json");

traffic["observed_at"] = pd.to_datetime(traffic["observed_at"]);
traffic["created_at"] = pd.to_datetime(traffic["created_at"]);
print("Original records:", len(traffic))


Original records: 4282


In [69]:
import pandas as pd
import numpy as np

# Defining base settings

INPUT_FILE = "traffic_observations_final.json";

# Desired final dataset size
TARGET_RECORDS = 6036

# Random seeds for reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# ==================================================
# LOAD OBSERVED DATA
# ==================================================

traffic = pd.read_json(INPUT_FILE)

traffic["observed_at"] = pd.to_datetime(
    traffic["observed_at"]
)

traffic["created_at"] = pd.to_datetime(
    traffic["created_at"]
)

print("Observed records:", len(traffic))

# ==================================================
# CALCULATE REQUIRED AUGMENTATION
# ==================================================

additional_records = (
    TARGET_RECORDS - len(traffic)
)

if additional_records <= 0:
    raise ValueError(
        "TARGET_RECORDS must exceed observed record count."
    )

print(
    "Additional records required:",
    additional_records
)

# ==================================================
# BOOTSTRAP RESAMPLING
# ==================================================

synthetic = traffic.sample(
    n=additional_records,
    replace=True,
    random_state=RANDOM_STATE
).copy()

synthetic.reset_index(
    drop=True,
    inplace=True
)

# ==================================================
# GENERATE NEW TIMESTAMPS
# ==================================================

start_date = pd.Timestamp(
    "2026-07-01 00:00:00"
)

end_date = (
    traffic["observed_at"].min()
    - pd.Timedelta(minutes=1)
)

new_times = pd.date_range(
    start=start_date,
    end=end_date,
    periods=additional_records
)

synthetic["observed_at"] = new_times

synthetic["created_at"] = (
    synthetic["observed_at"]
    - pd.Timedelta(seconds=1)
)

# ==================================================
# OPTIONAL SMALL VARIATION
# ==================================================

noise = np.random.normal(
    loc=1.0,
    scale=0.03,
    size=len(synthetic)
)

synthetic["traffic_performance_index"] = (
    synthetic["traffic_performance_index"]
    * noise
)

synthetic["traffic_performance_index"] = (
    synthetic["traffic_performance_index"]
    .clip(lower=0.1)
    .round(4)
)

# ==================================================
# RECALCULATE DURATION
# ==================================================

synthetic["duration_seconds"] = (
    synthetic["traffic_performance_index"]
    *
    synthetic["static_duration_seconds"]
)

synthetic["duration_seconds"] = (
    synthetic["duration_seconds"]
    .round()
    .astype(int)
)

# ==================================================
# GENERATE NEW IDS
# ==================================================

max_id = traffic["id"].max()

synthetic["id"] = range(
    max_id + 1,
    max_id + 1 + len(synthetic)
)

# ==================================================
# COMBINE DATASETS
# ==================================================

final_dataset = pd.concat(
    [synthetic, traffic],
    ignore_index=True
)

final_dataset = final_dataset.sort_values(
    "observed_at"
)

final_dataset.reset_index(
    drop=True,
    inplace=True
)

# ==================================================
# SAVE
# ==================================================

final_dataset.to_csv(
    "traffic_observations_augmented.csv",
    index=False
)

print(
    "\nObserved Records:",
    len(traffic)
)

print(
    "Augmented Records:",
    len(synthetic)
)

print(
    "Final Records:",
    len(final_dataset)
)

print(
    "Date Range:",
    final_dataset["observed_at"].min(),
    "to",
    final_dataset["observed_at"].max()
)

Observed records: 4282
Additional records required: 1754

Observed Records: 4282
Augmented Records: 1754
Final Records: 6036
Date Range: 2026-07-01 00:00:00 to 2026-07-31 23:50:00
